# 🏠 House Price Prediction — Preprocessing
## Objectif
Nettoyer et préparer les données pour la modélisation :
valeurs manquantes, outliers, encodage et normalisation.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Charger les données
df = pd.read_csv('../data/train.csv')
print("Données chargées :", df.shape)

Données chargées : (1460, 81)


In [2]:
# Supprimer les outliers détectés dans l'EDA
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000))]
print("Taille après suppression outliers :", df.shape)

Taille après suppression outliers : (1458, 81)


In [3]:
# Transformation logarithmique du prix
df['SalePrice'] = np.log(df['SalePrice'])
print("Prix transformé - moyenne :", round(df['SalePrice'].mean(), 2))
print("Prix transformé - std :", round(df['SalePrice'].std(), 2))

Prix transformé - moyenne : 12.02
Prix transformé - std : 0.4


In [5]:
# Variables où NaN veut dire "n'existe pas"
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in cols_none:
    df[col] = df[col].fillna('None')

# Variables numériques où NaN veut dire 0
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    df[col] = df[col].fillna(0)

# LotFrontage — on remplace par la médiane du quartier
df['LotFrontage'] = df.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

# Electrical — 1 seul manquant, on met le plus fréquent
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])

# Vérification
print("Valeurs manquantes restantes :", df.isnull().sum().sum())

Valeurs manquantes restantes : 0


In [6]:
# One-Hot Encoding des variables catégorielles
df = pd.get_dummies(df, drop_first=True)
print("Taille après encodage :", df.shape)

Taille après encodage : (1458, 260)


In [7]:
from sklearn.preprocessing import StandardScaler

# Séparer features et target
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features shape :", X_scaled.shape)
print("Target shape :", y.shape)

Features shape : (1458, 259)
Target shape : (1458,)


In [9]:
import os

# Créer le dossier processed
os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder les données preprocessées
X_df = pd.DataFrame(X_scaled, columns=X.columns)
X_df.to_csv('../data/processed/X_scaled.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)

print("Données sauvegardées !")

Données sauvegardées !
